# **Análise Algoritmos de Recomendação de Filmes**

A análise será conduzida para avaliar e comparar o desempenho de dois algoritmos de recomendação distintos: **User-Based** e **Content-Based**. Serão exploradas métricas como acurácia, precisão e revocação para medir a eficácia de cada modelo, possibilitando uma visão clara sobre a qualidade das recomendações geradas.

Import das bibliotecas utilizadas

In [1]:
# Responsável por manipular dados
import pandas as pd
# Responsável por implementar os algoritmos
import sklearn.metrics.pairwise as pw
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Leitura dos DataSet utilizados

In [2]:
# CSV Filmes
Filmes = pd.read_csv('movies.csv', sep = ',')
# CSV Ratings
Ratings = pd.read_csv('ratings.csv', sep = ',')
# CSV Links
Links = pd.read_csv('links.csv', sep = ',')
# CSV Tags
Tags = pd.read_csv('tags.csv', sep = ',')

In [17]:
# Impressão das primeiras cinco linhas do CSV Filmes
Filmes.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [18]:
# Impressão das primeiras cinco linhas do CSV Ratings
Ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [19]:
# Impressão das primeiras cinco linhas do CSV Links
Links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [20]:
# Impressão das primeiras cinco linhas do CSV Tags
Tags.head()

,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [3]:
# Juntado o CSV Ratings ao CSV Filmes
df = Filmes.merge(Ratings, on = 'movieId')
df.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,10,2.5,1169265231
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,11,3.0,850085076
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.0,1027305751
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,19,3.0,974704488
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,20,5.0,1553184230


# **Implementação do Algoritmo Baseado em Usuário (User-Based)**

O algoritmo de recomendação baseado em usuários (User-Based) utiliza as avaliações atribuídas por cada usuário para construir um perfil de preferência. Com isso, é possível mapear um conjunto de filmes que um usuário específico provavelmente se interessará em assistir, com base nas semelhanças com outros usuários.

In [4]:
# Criação da tabela do DataSet Filmes(já agregado com o Ratings) para manipulação no algoritmo
tabela_filmes = pd.pivot_table(df, index='title', columns='userId', values='rating').fillna(0)
tabela_filmes.head()

C:\Users\felip\AppData\Local\Temp\ipykernel_1728\949495065.py:2: PerformanceWarning: The following operation may generate 16927658572 cells in the resulting pandas object.
  tabela_filmes = pd.pivot_table(df, index='title', columns='userId', values='rating').fillna(0)


MemoryError: Unable to allocate 15.8 GiB for an array with shape (16927658572,) and data type bool

In [ ]:
rec = pw.cosine_similarity(tabela_filmes)
rec_df = pd.DataFrame(rec, columns=tabela_filmes.index, index=tabela_filmes.index)
rec_df.head()

In [ ]:
cossine_df = pd.DataFrame(rec_df['Toy Story'].sort_values(ascending=False))
cossine_df.columns = ['Recomendações']
cossine_df.head(20)

In [ ]:
cossine_df.to_csv('Recomendações(Cosseno).csv')

# **Implementação do Algoritmo Baseado no Contéudo de cada Filme**

O algoritmo de recomendação baseado em conteúdo (Content-Based) utiliza as características dos filmes, como gênero, diretor, elenco, entre outros atributos, para construir um perfil de preferências de cada usuário. A partir desse perfil, o algoritmo identifica e recomenda filmes com características semelhantes aos filmes previamente bem avaliados pelo usuário.

In [ ]:
dados = pd.read_csv('Dados.csv', sep=',')
dados.head()

In [ ]:
dados.shape

In [ ]:
Filmes['movieId'] = Filmes['movieId'].apply(lambda x: str(x))

In [ ]:
df2 = Filmes.merge(dados, left_on='title', right_on='Name', how='left')
df2['movieId'] = df2['movieId'].astype(int)
Tags['movieId'] = Tags['movieId'].astype(int)
df2 = df2.merge(Tags, left_on='movieId', right_on='movieId', how='left')
df2['Infos'] = df2['genres'] + str(df2['Directors_Cast']) + str(df2['Discription']) + df2['tag']
df2.head()

In [ ]:
vec = TfidfVectorizer()
tfidf = vec.fit_transform(df2['Infos'].apply(lambda x: np.str_(x)))

In [ ]:
sim = cosine_similarity(tfidf)
sim

In [ ]:
sim_df2 = pd.DataFrame(sim, columns=df2['title'], index=df2['title'])
sim_df2.head()

In [ ]:
final_df = pd.DataFrame(sim_df2['La La Land'].sort_values(ascending=False))
final_df.columns = ['Recomendações']
final_df.head(20)

In [ ]:
final_df.to_csv('Recomendações(Seno).csv')

# **Análise Exploratória dos Dados**


Análise de Dados dos Algoritmos de Recomendação Baseado em Usuário e em Conteúdo
Após a implementação dos algoritmos de recomendação Baseado em Usuário (User-Based) e Baseado em Conteúdo (Content-Based), é possível realizar uma análise detalhada dos dados gerados para avaliar a eficácia e a precisão de cada modelo. A análise incluirá métricas como acurácia, precisão, revocação e outras que possam oferecer insights sobre a qualidade das recomendações fornecidas pelos modelos.

Import das bibliotecas utilizadas

In [ ]:
# Responsável por plotar gráficos
import matplotlib as plt
# Responsável por ferramentas matematicas
import numpy as np
# Responsável por ferramentas estatísticas
import seaborn as sns

**Análise 1: Vamos análisar primeiro a distribuição das pontuações de recomendação nos dois algoritmos**

Histograma das pontuações de recomendação para usuários(user-based)

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(cossine_df['score'], kde=True, bins=30, color='green')
plt.title("Distribuição de Recomendações Baseadas no Usuário (Cosseno)")
plt.xlabel("Pontuação de Recomendação")
plt.ylabel("Frequência")
plt.show()

Histograma das pontuações de recomendação para filmes(Content-based)

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(final_df['score'], kde=True, bins=30, color='blue')
plt.title("Distribuição de Recomendações Baseadas no Conteúdo (Seno)")
plt.xlabel("Pontuação de Recomendação")
plt.ylabel("Frequência")
plt.show()

**Análise 2: Comparação de recomendações para usuários mais ativos(User-based) **



In [ ]:
top_usuarios = cossine_df['userId'].value_counts().head(10).index
df_top_usuarios = cossine_df[cossine_df['userId'].isin(top_usuarios)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_top_usuarios, x='userId', y='score', palette="Set2")
plt.title("Pontuações de Recomendação por Usuário Ativo (Cosseno)")
plt.xlabel("Usuário")
plt.ylabel("Pontuação de Recomendação")
plt.show()

**Análise 3: Comparação de recomendações para os gêneros mais populares(Content-based)**



In [ ]:
if 'genre' in final_df.columns:
    top_generos = final_df['genre'].value_counts().head(10).index
    df_top_generos = final_df[final_df['genre'].isin(top_generos)]

    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df_top_generos, x='genre', y='score', palette="Set3")
    plt.title("Pontuações de Recomendação por Gênero (Seno)")
    plt.xlabel("Gênero")
    plt.ylabel("Pontuação de Recomendação")
    plt.xticks(rotation=45)
    plt.show()

**Análise 4: Outliers em ambos os métodos**

Outliers em recomendações baseadas no usuário(User-based)

In [ ]:
outliers_cossine = cossine_df[cossine_df['score'] > cossine_df['score'].quantile(0.95)]
print("\nOutliers - Recomendações Baseadas no Usuário (Cosseno):")
print(outliers_cossine)

Outliers em Recomendações Baseadas no Conteúdo(Content-based)

In [ ]:
outliers_seno = final_df[final_df['score'] > final_df['score'].quantile(0.95)]
print("\nOutliers - Recomendações Baseadas no Conteúdo (Seno):")
print(outliers_seno)